In [11]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer 
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import joblib

# Загружаем сплит
X_train = pd.read_csv('../data/processed/X_train.csv')
X_val = pd.read_csv('../data/processed/X_val.csv')
X_test = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_val = pd.read_csv('../data/processed/y_val.csv').squeeze()
y_test = pd.read_csv('../data/processed/y_test.csv').squeeze()

# Загружаем препроцессор (если он уже сохранён)
# Но т.к. в нём может не быть импутации, проще создать новый с импутацией
# Определяем числовые и категориальные колонки (можно взять из X_train)
numeric_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

# Создаём новый препроцессор с импутацией
# Для числовых: заполняем медианой
# Для категориальных: заполняем самой частой категорией (most_frequent)
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),   # или 'mean'
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, categorical_cols)
])

# Создаём пайплайн
baseline_pipe = Pipeline([
    ('prep', preprocessor),
    ('reg', LinearRegression())
])

# Обучение
baseline_pipe.fit(X_train, y_train)

# Предсказания
y_pred_train = baseline_pipe.predict(X_train)
y_pred_val = baseline_pipe.predict(X_val)
y_pred_test = baseline_pipe.predict(X_test)

# Метрики
def print_metrics(y_true, y_pred, name):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"{name}: R²={r2:.4f}, MAE={mae:.2f}, RMSE={rmse:.2f}")

print_metrics(y_train, y_pred_train, "Train")
print_metrics(y_val, y_pred_val, "Val")
print_metrics(y_test, y_pred_test, "Test")

# Сохраняем модель
joblib.dump(baseline_pipe, '../models/baseline_model.pkl')

/var/folders/t9/w50p8w0917b0t61ks_6y_3g40000gn/T/ipykernel_24740/2886571032.py:24: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()


Train: R²=0.9502, MAE=1.00, RMSE=1.23
Val: R²=0.9501, MAE=1.00, RMSE=1.23
Test: R²=0.9517, MAE=1.00, RMSE=1.22


['../models/baseline_model.pkl']